# Week 2 — Operational Data Cleaner & Insight Generator
### Fuel Processing Plant — Sensor Log Analysis

---
**Dataset:** `ops_sensor_log_dirty.csv`
**Primary metric:** `pressure_bar` — pipeline pressure in bar  
**Objective:** Build a reusable cleaning pipeline, perform time-series analysis, and surface hidden operational patterns invisible in the raw data.


#### 0. Environment Setup

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Optional — visual null heatmap
try:
    import missingno as msno
    HAS_MISSINGNO = True
except ImportError:
    HAS_MISSINGNO = False
    print("missingno not installed — pip install missingno to enable visual null map")

print("Libraries loaded ✓")


missingno not installed — pip install missingno to enable visual null map
Libraries loaded ✓


### 1. Ingestion & Data Health Report

loading the raw CSV and running three diagnostic tools:
- `.info()` — column types and null counts at a glance
- `.describe()` — statistical distribution of each numeric column
- `missingno` — visual pattern of where nulls cluster (if installed)


In [17]:
df_raw = pd.read_csv("ops_sensor_log_dirty.csv")

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print()
df_raw.info()


Shape: 1,018 rows × 6 columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1018 entries, 0 to 1017
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      1018 non-null   object 
 1   pressure_bar   980 non-null    float64
 2   temperature_c  988 non-null    float64
 3   flow_rate_m3h  993 non-null    float64
 4   zone           1018 non-null   object 
 5   operator_id    1018 non-null   object 
dtypes: float64(3), object(3)
memory usage: 47.8+ KB


In [ ]:
df_raw.describe().round(3)


,pressure_bar,temperature_c,flow_rate_m3h
count,980.000,988.000,993.000
mean,5.822,71.907,320.569
std,11.443,5.793,30.760
min,-9.900,60.560,251.840
25%,3.993,66.376,294.450
50%,4.220,72.083,321.082
75%,4.522,77.474,347.785
max,99.500,82.940,390.928


In [ ]:
if HAS_MISSINGNO:
    fig, ax = plt.subplots(figsize=(10, 4))
    msno.matrix(df_raw, ax=ax, sparkline=False, fontsize=9)
    ax.set_title("Missing Value Map — ops_sensor_log_dirty.csv", fontsize=11, pad=10)
    plt.tight_layout()
    plt.show()
else:
    print("Null counts per column:")
    print(df_raw.isnull().sum())


Null counts per column:
timestamp         0
pressure_bar     38
temperature_c    30
flow_rate_m3h    25
zone              0
operator_id       0
dtype: int64


###  Data Health Report

After profiling the raw dataset (**1,018 rows × 6 columns**), I identified **five specific data quality issues**:

---

**Issue 1 — Unparseable Timestamps (literal string `"NaT"`)**  
The `timestamp` column arrives as plain text. Six rows contain the literal string `"NaT"` rather than a valid datetime. These rows have no recoverable position in the time series and must be dropped entirely before any time-based operation. All other timestamps parse cleanly.

---

**Issue 2 — Physically Impossible Pressure Readings (18 values)**  
`pressure_bar` ranges from **-9.9** to **99.5** in the raw data. The operating envelope for this plant is 0–15 bar. Exactly 18 readings breach this boundary (16 above 15 bar, 2 below 0). These are not operational events — they are sensor transmission faults. Left in place, they inflate the standard deviation to 11.4 and make the mean (5.82) meaningless, concealing the true signal entirely.

---

**Issue 3 — Missing Sensor Values (~9% of cells)**  
`pressure_bar` has 38 nulls, `temperature_c` has 30, `flow_rate_m3h` has 25. Because this is a continuous 10-minute time series, **dropping** these rows would create irregular time gaps that distort resampling and rolling calculations. **Linear interpolation** is the correct fix — it estimates each missing value from its immediate neighbours, which is physically justified for sensor readings that change gradually.

---

**Issue 4 — Duplicate Rows (10 exact duplicates)**  
Ten rows are exact copies of earlier readings, arising from data pipeline retransmission errors. Keeping them would double-count those time windows in any aggregation.

---

## 2. Cleaning Pipeline

The function below encapsulates all five fixes in a strict, ordered sequence.  
Order matters: timestamps must be parsed before we can sort chronologically;  
duplicates must be removed before interpolation (to avoid interpolating between two identical readings).

In [18]:
def clean_ops_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans a raw operational sensor log DataFrame.

    Processing steps (in order):
      1. Parse 'timestamp' column to datetime; drop rows where timestamp is unrecoverable.
      2. Sort chronologically and reset the integer index.
      3. Remove exact duplicate rows.
      4. Standardise zone labels to consistent 'ZONE X' format.
      5. Null-out physically impossible pressure readings (< 0 or > 15 bar).
      6. Linearly interpolate all numeric sensor columns to fill remaining gaps.
      7. Set timestamp as the DataFrame index for time-series operations.

    Parameters
    ----------
    df : pd.DataFrame
        Raw sensor log as loaded from CSV. Must contain columns:
        'timestamp', 'pressure_bar', 'temperature_c', 'flow_rate_m3h', 'zone'.

    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with DatetimeIndex, zero nulls in sensor columns.
    """
    df = df.copy()   # never mutate the original

    # ── Step 1: Parse timestamps ──────────────────────────────────────────────
    # errors='coerce' turns unparseable strings (like literal "NaT") into NaT
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    n_bad_ts = df["timestamp"].isna().sum()
    df = df.dropna(subset=["timestamp"])
    print(f"  Step 1  Dropped {n_bad_ts} unparseable timestamp rows")

    # ── Step 2: Sort chronologically ─────────────────────────────────────────
    df = df.sort_values("timestamp").reset_index(drop=True)
    print(f"  Step 2  Sorted {len(df):,} rows chronologically")

    # ── Step 3: Remove duplicates ─────────────────────────────────────────────
    n_before = len(df)
    df = df.drop_duplicates()
    print(f"  Step 3   Removed {n_before - len(df)} duplicate rows")

    # ── Step 4: Standardise zone labels ──────────────────────────────────────
    # .upper() collapses case; .replace() removes hyphens/underscores
    df["zone"] = (
        df["zone"]
        .str.strip()
        .str.upper()
        .str.replace(r"[-_]", " ", regex=True)
    )
    print(f"  Step 4  Zone labels normalised → {sorted(df['zone'].unique())}")

    # ── Step 5: Null-out impossible pressure values ───────────────────────────
    # Physical ceiling: 15 bar. Values outside 0–15 are sensor faults, not events.
    impossible = (df["pressure_bar"] < 0) | (df["pressure_bar"] > 15)
    df.loc[impossible, "pressure_bar"] = np.nan
    print(f"  Step 5  Nulled {impossible.sum()} impossible pressure readings")

    # ── Step 6: Interpolate remaining nulls ───────────────────────────────────
    sensor_cols = ["pressure_bar", "temperature_c", "flow_rate_m3h"]
    for col in sensor_cols:
        n_null = df[col].isna().sum()
        df[col] = df[col].interpolate(method="linear", limit_direction="both")
        print(f"  Step 6   Interpolated {n_null:>2} nulls in '{col}'")

    # ── Step 7: Set DatetimeIndex ─────────────────────────────────────────────
    df = df.set_index("timestamp")
    print(f"\n Cleaning complete — final shape: {df.shape}")
    return df


print("Running clean_ops_data() on raw dataset...\n")
df_clean = clean_ops_data(df_raw)


Running clean_ops_data() on raw dataset...

  Step 1  Dropped 6 unparseable timestamp rows
  Step 2  Sorted 1,012 rows chronologically
  Step 3   Removed 10 duplicate rows
  Step 4  Zone labels normalised → ['ZONE A', 'ZONE B', 'ZONE C']
  Step 5  Nulled 18 impossible pressure readings
  Step 6   Interpolated 56 nulls in 'pressure_bar'
  Step 6   Interpolated 30 nulls in 'temperature_c'
  Step 6   Interpolated 25 nulls in 'flow_rate_m3h'

 Cleaning complete — final shape: (1002, 5)


## 3. Time-Series Analysis

We resample the 10-minute readings to **hourly means** — this smooths micro-fluctuations and aligns with operational shift handover cadence.

The **24-hour rolling average** acts as a signal extractor: it filters out day-to-day noise and reveals any sustained directional drift. A rolling average that trends upward across multiple days indicates a systemic issue, not random variation.


In [ ]:
def assign_shift(hour: int) -> str:
    """Map hour-of-day (0–23) to a named operational shift."""
    if 6 <= hour < 14:
        return "Morning"
    elif 14 <= hour < 22:
        return "Afternoon"
    else:
        return "Night"


# Assign shift on the 10-minute data (before resampling loses the hour info)
df_clean["shift"] = df_clean.index.hour.map(assign_shift)

# Hourly resample — mean of all readings within each clock hour
hourly = df_clean[["pressure_bar", "temperature_c", "flow_rate_m3h"]].resample("h").mean()

# 24-hour rolling average on the primary KPI (pressure)
hourly["pressure_rolling24"] = hourly["pressure_bar"].rolling(window=24, min_periods=1).mean()

print(f"Resampled to {len(hourly)} hourly intervals")
print(f"Period: {hourly.index[0]}  →  {hourly.index[-1]}")
print()
hourly.head(10).round(3)
